# BrainTart - Attention U-Net 3D for BraTS 2026 Inpainting

This notebook clones the BrainTart repository, downloads the challenge data,
trains the Attention U-Net 3D model, runs inference, and evaluates locally.

**Hardware target:** 2x T4 (16 GB each) on Kaggle, PyTorch DDP.

### Sections
1. Environment Setup & Repo Clone
2. Synapse Auth & Data Download
3. Configuration Overview
4. Training
5. Inference & Submission Generation
6. Local Evaluation
7. Submission Checklist

## 1. Environment Setup & Repo Clone

In [1]:
# Install dependencies and clone the BrainTart repo
import subprocess, sys, os

# Clone the repo (replace with your actual repo URL)
REPO_URL = "https://github.com/PaulOkwija/BrainTart"  # TODO: set your repo URL
REPO_DIR = "/kaggle/working/BrainTart"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    print(f"Cloned BrainTart to {REPO_DIR}")
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
    print(f"Updated BrainTart at {REPO_DIR}")

# Install Python dependencies
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r",
     os.path.join(REPO_DIR, "requirements.txt")]
)
print("Dependencies installed.")

Cloning into '/kaggle/working/BrainTart'...


Cloned BrainTart to /kaggle/working/BrainTart


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.8/836.8 kB 41.4 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.0 MB/s eta 0:00:00


Dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=

In [2]:
# Add repo to sys.path so imports work
import sys
sys.path.insert(0, REPO_DIR)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {props.name}  {props.total_memory/1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPUs     : 2
  GPU 0 : Tesla T4  15.6 GB
  GPU 1 : Tesla T4  15.6 GB


## 2. Synapse Auth & Data Download

In [3]:
import zipfile
from pathlib import Path
import synapseclient
from kaggle_secrets import UserSecretsClient

SYNAPSE_TOKEN = UserSecretsClient().get_secret("brats")
syn = synapseclient.Synapse()
syn.login(authToken=SYNAPSE_TOKEN)
print("Logged in to Synapse.")

TRAINING_ENTITY   = "syn51523038"
VALIDATION_ENTITY = "syn51684975"  # TODO: verify on Synapse

DOWNLOAD_DIR = Path("/kaggle/working/brats_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

Welcome, Adelois!



Logged in to Synapse.


In [4]:
# Download and extract TRAINING data
TRAIN_ROOT = Path("/kaggle/working/brats_data_train")
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

if not any(TRAIN_ROOT.rglob("BraTS-GLI-*-*")):
    print("Downloading training data...")
    entity = syn.get(entity=TRAINING_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))
    print(f"Downloaded to: {entity.path}")
    print("Extracting...")
    with zipfile.ZipFile(entity.path, "r") as zf:
        zf.extractall(TRAIN_ROOT)
    candidates = list(TRAIN_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        TRAIN_ROOT = candidates[0].parent
else:
    candidates = list(TRAIN_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        TRAIN_ROOT = candidates[0].parent
    print("Training data already extracted.")

train_cases = [p for p in TRAIN_ROOT.iterdir() if p.is_dir()]
print(f"TRAIN_ROOT   : {TRAIN_ROOT}")
print(f"Train cases  : {len(train_cases)}")

[WARNING] /tmp/ipykernel_23/2203499290.py:7: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  entity = syn.get(entity=TRAINING_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))



[syn51523038]: Downloaded to /kaggle/working/brats_download/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Training.zip


Downloaded to: /kaggle/working/brats_download/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Training.zip
Extracting...


TRAIN_ROOT   : /kaggle/working/brats_data_train/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Training
Train cases  : 1251


In [5]:
# Download and extract VALIDATION data
VAL_ROOT = Path("/kaggle/working/brats_data_val")
VAL_ROOT.mkdir(parents=True, exist_ok=True)

if not any(VAL_ROOT.rglob("BraTS-GLI-*-*")):
    print("Downloading validation data...")
    try:
        entity = syn.get(entity=VALIDATION_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))
        print(f"Downloaded to: {entity.path}")
        print("Extracting...")
        with zipfile.ZipFile(entity.path, "r") as zf:
            zf.extractall(VAL_ROOT)
        candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
        if candidates:
            VAL_ROOT = candidates[0].parent
        val_cases = [p for p in VAL_ROOT.iterdir() if p.is_dir()]
        print(f"VAL_ROOT     : {VAL_ROOT}")
        print(f"Val cases    : {len(val_cases)}")
    except Exception as e:
        print(f"Could not download validation data: {e}")
        print("Check VALIDATION_ENTITY ID or download manually from Synapse.")
        print("Falling back to training data for inference demo.")
        VAL_ROOT = TRAIN_ROOT
else:
    candidates = list(VAL_ROOT.rglob("BraTS-GLI-*-*"))
    if candidates:
        VAL_ROOT = candidates[0].parent
    val_cases = [p for p in VAL_ROOT.iterdir() if p.is_dir()]
    print("Validation data already extracted.")
    print(f"VAL_ROOT     : {VAL_ROOT}")
    print(f"Val cases    : {len(val_cases)}")

[WARNING] /tmp/ipykernel_23/3974315982.py:8: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  entity = syn.get(entity=VALIDATION_ENTITY, version=1, downloadLocation=str(DOWNLOAD_DIR))



[syn51684975]: Downloaded to /kaggle/working/brats_download/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Validation.zip


Downloaded to: /kaggle/working/brats_download/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Validation.zip
Extracting...


VAL_ROOT     : /kaggle/working/brats_data_val/ASNR-MICCAI-BraTS2023-Local-Synthesis-Challenge-Validation
Val cases    : 219


## 3. Configuration Overview

In [6]:
import shutil

# Delete a specific folder inside the output directory
shutil.rmtree("/kaggle/working/brats_download", ignore_errors=True)


In [7]:
from configs import Config
DATASET_ROOT = '/kaggle/working/brats_data_train'
cfg = Config()
cfg.DATASET_ROOT = DATASET_ROOT
cfg.makedirs()

# Override any defaults here:
# cfg.NUM_EPOCHS = 100
# cfg.BATCH_PER_GPU = 1

print(f"Crop shape     : {cfg.CROP_SHAPE}")
print(f"Base channels  : {cfg.BASE_CHANNELS}")
print(f"Depth          : {cfg.DEPTH}")
print(f"Effective batch: {cfg.BATCH_PER_GPU * max(torch.cuda.device_count(), 1)}")
print(f"Epochs         : {cfg.NUM_EPOCHS}")

Crop shape     : (96, 96, 96)
Base channels  : 32
Depth          : 3
Effective batch: 4
Epochs         : 60


## 4. Training

Runs DDP on available GPUs via the `train.py` script.

In [8]:
import subprocess, sys

train_cmd = [
    sys.executable, f"{REPO_DIR}/train.py",
    "--dataset", str(DATASET_ROOT),
    "--epochs", str(cfg.NUM_EPOCHS),
    "--batch", str(cfg.BATCH_PER_GPU),
    "--lr", str(cfg.LR),
    "--seed", str(cfg.SEED),
    "--crop", str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
    "--base-ch", str(cfg.BASE_CHANNELS),
    "--depth", str(cfg.DEPTH),
    "--checkpoint-dir", str(cfg.CHECKPOINT_DIR),
    "--results-dir", str(cfg.RESULTS_DIR),
    "--output-dir", str(cfg.OUTPUT_DIR),
]

print("Running:", " ".join(train_cmd))
result = subprocess.run(train_cmd, capture_output=False)
print(f"Training exited with code {result.returncode}")

Running: /usr/bin/python3 /kaggle/working/BrainTart/train.py --dataset /kaggle/working/brats_data_train --epochs 60 --batch 2 --lr 0.0002 --seed 2023 --crop 96 96 96 --base-ch 32 --depth 3 --checkpoint-dir /kaggle/working/checkpoints --results-dir /kaggle/working/results --output-dir /kaggle/working/outputs


[TrainDataset] 1251 cases | augment=True | cache=/kaggle/working/.patch_cache
Train cases: 1063 | Val cases: 188
Launching DDP training on 2 GPUs...


[GPU0] Ep 1/60:   0%|          | 0/266 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


[GPU0] Ep 1/60:   0%|          | 1/266 [01:03<4:40:57, 63.61s/it, l1=0.6913, loss=1.9820]

...

[GPU0] Ep 12/60:  29%|██▊       | 76/266 [04:45<11:49,  3.74s/it, l1=0.0946, loss=0.2117]

[GPU0] Ep 12/60:  29%|██▉       | 77/266 [04:48<11:45,  3.73s/it, l1=0.0633, loss=0.1401]

[GPU0] Ep 12/60:  29%|██▉       | 78/266 [04:52<11:42,  3.74s/it, l1=0.1049, loss=0.2373]

[GPU0] Ep 12/60:  30%|██▉       | 79/266 [04:56<11:38,  3.73s/it, l1=0.0735, loss=0.1449]

[GPU0] Ep 12/60:  30%|███       | 80/266 [05:00<11:34,  3.74s/it, l1=0.0949, loss=0.1942]

In [ ]:
# Display the training curve
from IPython.display import Image, display
from pathlib import Path

curve_path = cfg.OUTPUT_DIR / "training_curve_final.png"
if curve_path.exists():
    display(Image(filename=str(curve_path)))
else:
    print("Training curve not found — check training output above.")

## 5. Inference & Submission Generation

Output format: `BraTS-GLI-XXXXX-YYY-t1n-inference.nii.gz` (240x240x155)

In [ ]:
import subprocess, sys

checkpoint = cfg.CHECKPOINT_DIR / "best_model.pt"
if not checkpoint.exists():
    # Fallback to latest periodic checkpoint
    ckpts = sorted(cfg.CHECKPOINT_DIR.glob("ckpt_epoch_*.pt"))
    checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    infer_cmd = [
        sys.executable, f"{REPO_DIR}/inference.py",
        "--dataset", str(DATASET_ROOT),
        "--checkpoint", str(checkpoint),
        "--results-dir", str(cfg.RESULTS_DIR),
        "--crop", str(cfg.CROP_SHAPE[0]), str(cfg.CROP_SHAPE[1]), str(cfg.CROP_SHAPE[2]),
        "--base-ch", str(cfg.BASE_CHANNELS),
        "--depth", str(cfg.DEPTH),
    ]
    print("Running:", " ".join(infer_cmd))
    subprocess.run(infer_cmd)
else:
    print("ERROR: No checkpoint found. Train first.")

## 6. Local Evaluation

Mirrors the Synapse server evaluation: SSIM, PSNR, MSE on mask-healthy region,
normalised by max(t1n_voided).

In [ ]:
import subprocess, sys

eval_cmd = [
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--max-cases", "30",  # Quick sanity check; remove for full eval
]
print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd)

## 7. Submission Checklist & Upload

Run this before uploading to Synapse.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, f"{REPO_DIR}/evaluate.py",
    "--dataset", str(DATASET_ROOT),
    "--results", str(cfg.RESULTS_DIR),
    "--checklist",
])

print("\nTo upload to Synapse:")
print(f"  cd {cfg.RESULTS_DIR}")
print(f"  zip -j results.zip *.nii.gz")
print("  Then upload results.zip on the Synapse challenge page.")